# Phase 4 — Feature Sanity Review (the gate)

Before a single rating is computed we eyeball the raw features. The question
is simple and human: **do the leaderboards contain the players football fans
would expect?** If the top finishers aren't the famous strikers, something is
wrong with the data engineering and no clever rating maths will fix it.

Each leaderboard ranks **rated players with ≥ 1000 minutes** (≈ 11 full
matches — enough that per-90 rates are stable) by the *primary raw signal* for
that attribute. These are raw per-90 numbers, **not** the final 0–99 ratings.

In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import wyscout_lib as wl

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

career = pd.read_parquet(wl.DATA / "player_career_stats.parquet")
pool = career[(career.is_rated) & (career.minutes_played >= 1000)].copy()
print(f"leaderboard pool: {len(pool):,} players with >= 1000 minutes")


def board(metric, n=25, position=None, extra=None, title=""):
    df = pool if position is None else pool[pool.position_code == position]
    cols = ["short_name", "position_code", "nationality", "matches_played", metric]
    if extra:
        cols += [c for c in extra if c not in cols]
    out = (df.nlargest(n, metric)[cols]
             .round(3).reset_index(drop=True))
    out.index = out.index + 1
    print(f"\n{'='*70}\nTOP {n} — {title or metric}\n{'='*70}")
    return out

leaderboard pool: 1,577 players with >= 1000 minutes


## 1. FINISHING — goals per 90
The acid test. Expect Messi, Ronaldo, Mbappé, Kane, Salah, Cavani, Immobile,
Aubameyang, Icardi, Lewandowski near the top.

In [2]:
board("goals_p90", position=None, extra=["goals", "shots_total", "shot_accuracy_pct"],
      title="FINISHING — goals per 90 (all positions)")


TOP 25 — FINISHING — goals per 90 (all positions)


,short_name,position_code,nationality,matches_played,goals_p90,goals,shots_total,shot_accuracy_pct
1,Mohamed Salah,FW,Egypt,38,0.986,34,141,0.475
2,S. Agüero,FW,Spain,29,0.965,23,92,0.446
3,E. Cavani,FW,Italy,36,0.952,31,107,0.505
4,L. Messi,FW,Spain,40,0.939,35,154,0.468
5,C. Immobile,FW,Italy,35,0.937,29,104,0.462
6,R. Lewandowski,FW,Poland,38,0.915,30,124,0.435
7,Cristiano Ronaldo,FW,Portugal,38,0.905,33,196,0.418
8,Neymar,FW,Brazil,25,0.881,22,94,0.457
9,M. Icardi,FW,Italy,34,0.880,29,87,0.506
10,P. Aubameyang,FW,Gabon,29,0.836,23,73,0.548


## 2. CREATIVITY — key passes per 90 (shot-assist passes)
Expect De Bruyne, Messi, Neymar, Özil, David Silva, Pjanić, Mertens.

In [3]:
board("key_passes_p90", extra=["assists", "assists_p90", "smart_passes_p90"],
      title="CREATIVITY — key passes per 90")


TOP 25 — CREATIVITY — key passes per 90


,short_name,position_code,nationality,matches_played,key_passes_p90,assists,assists_p90,smart_passes_p90
1,D. Payet,MD,France,38,1.352,14,0.440,3.301
2,Neymar,FW,Brazil,25,1.242,13,0.521,6.008
3,Lucas Vázquez,FW,Spain,35,1.169,6,0.305,1.424
4,Marcelo,DF,Spain,32,1.168,6,0.212,2.017
5,G. Bale,FW,Wales,32,1.157,2,0.077,1.003
6,A. Sánchez,FW,Chile,31,1.093,6,0.212,4.794
7,A. Gómez,FW,Italy,33,1.078,9,0.294,1.928
8,L. Insigne,FW,Italy,40,1.033,9,0.258,2.668
9,L. Messi,FW,Spain,40,0.965,14,0.375,4.184
10,M. Özil,MD,Germany,34,0.958,9,0.278,3.122


## 3. PASSING — progressive passes per 90 (with completion as context)
Expect deep distributors & ball-playing centre-backs: Jorginho, Kroos,
Alonso, Busquets, Verratti, Bonucci, Alaba.

In [4]:
board("progressive_passes_p90", extra=["pass_completion_pct", "pass_total_p90"],
      title="PASSING — progressive passes per 90")


TOP 25 — PASSING — progressive passes per 90


,short_name,position_code,nationality,matches_played,progressive_passes_p90,pass_completion_pct,pass_total_p90
1,J. Boateng,DF,Germany,27,37.410,0.880,77.917
2,M. Hummels,DF,Germany,32,34.571,0.904,76.256
3,Albiol,DF,Spain,31,34.037,0.913,78.446
4,K. Koulibaly,DF,Senegal,38,33.669,0.922,80.018
5,Sergio Ramos,DF,Spain,34,33.228,0.927,75.814
6,Jorginho,MD,Italy,33,29.560,0.900,105.457
7,Fàbregas,MD,Spain,36,29.554,0.855,79.954
8,Ö. Toprak,DF,Turkey,26,29.301,0.921,77.245
9,T. Alderweireld,DF,Belgium,25,28.380,0.890,64.446
10,É. Banega,MD,Spain,34,28.358,0.869,77.148


## 4. DEFENDING — defensive engagement (defensive duels + clearances per 90)
Volume-based by design (duel *win* tags are unreliable). Expect centre-backs &
defensive midfielders: Kanté, defensive CBs, full-backs.

In [5]:
pool["def_engagement_p90"] = pool["defensive_duels_p90"] + pool["clearances_p90"]
board("def_engagement_p90", extra=["defensive_duels_p90", "clearances_p90", "aerial_duels_p90"],
      title="DEFENDING — defensive duels + clearances per 90")


TOP 25 — DEFENDING — defensive duels + clearances per 90


,short_name,position_code,nationality,matches_played,def_engagement_p90,defensive_duels_p90,clearances_p90,aerial_duels_p90
1,Víctor Sánchez,MD,Spain,30,14.915,12.904,2.011,3.509
2,William,DF,Brazil,22,14.619,12.331,2.288,3.348
3,A. Mbengue,DF,Senegal,19,14.436,12.801,1.635,2.368
4,L. Balogun,DF,Nigeria,17,14.191,10.581,3.610,5.539
5,W. Ndidi,MD,Nigeria,36,14.115,12.297,1.818,7.043
6,F. Guilbert,DF,France,36,14.000,12.028,1.972,5.194
7,V. Behrami,MD,Switzerland,28,13.940,12.807,1.133,2.614
8,F. Sørensen,DF,Denmark,28,13.922,9.746,4.177,5.569
9,F. Depaoli,MD,Italy,19,13.829,11.486,2.343,4.081
10,K. Laimer,MD,Austria,22,13.716,12.500,1.216,3.581


## 5. WORK RATE — total involvement per 90
Expect high-volume midfielders and full-backs who touch the ball constantly.

In [6]:
board("total_events_p90", extra=["duels_total_p90", "touches_p90"],
      title="WORK RATE — total events per 90")


TOP 25 — WORK RATE — total events per 90


,short_name,position_code,nationality,matches_played,total_events_p90,duels_total_p90,touches_p90
1,M. Verratti,MD,Italy,22,135.601,22.200,5.641
2,Jorginho,MD,Italy,33,129.538,15.829,4.160
3,Isco,MD,Spain,34,127.413,26.439,8.896
4,Thiago Alcântara,MD,Brazil,23,125.088,21.740,5.220
5,T. Motta,MD,Italy,23,122.397,18.973,4.452
6,M. Lopez,MD,Algeria,24,121.202,17.961,6.096
7,Mário Rui,DF,Portugal,25,120.230,18.328,6.037
8,É. Banega,MD,Spain,34,119.736,24.602,5.134
9,Fernandinho,MD,Brazil,38,119.348,19.694,7.048
10,Neymar,FW,Brazil,25,119.159,34.726,6.689


## 6. SHOT STOPPING — goalkeepers by save %
(min 1000 minutes). Expect the elite league keepers at the top.

In [7]:
board("save_pct", position="GK", extra=["saves", "conceded", "saves_p90"],
      title="SHOT STOPPING — save % (GK only)")


TOP 25 — SHOT STOPPING — save % (GK only)


,short_name,position_code,nationality,matches_played,save_pct,saves,conceded,saves_p90
1,J. Oblak,GK,Slovenia,37,0.862,137,22,3.703
2,G. Buffon,GK,Italy,25,0.850,91,16,3.635
3,M. Neuer,GK,Germany,12,0.847,50,9,4.054
4,Alisson,GK,Germany,42,0.836,158,31,3.762
5,M. ter Stegen,GK,Germany,37,0.827,134,28,3.622
6,Neto,GK,Brazil,33,0.823,153,33,4.636
7,N. Pope,GK,England,35,0.822,162,35,4.682
8,David de Gea,GK,Spain,45,0.822,175,38,3.860
9,R. Gurtner,GK,France,37,0.816,182,41,4.919
10,S. Handanovič,GK,Slovenia,38,0.814,131,30,3.447


## Verdict
If these six leaderboards read like a list of the players you'd expect, the
feature layer is trustworthy and we can proceed to **Phase 5 — attribute
construction, percentiles, confidence, and archetypes**.